In [1]:
from pyspark.sql import SparkSession

# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("MAST30034 Tutorial 1")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/18 17:24:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/18 17:24:56 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/09/18 17:24:56 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [2]:
# Reading in the external ABS data as business indicators  
from urllib.request import urlretrieve
import os

# SEIFA 2021, SA2 level -- gives IRSAD score/decile per SA2 (socio-economic index)
# source: https://www.abs.gov.au/statistics/people/people-and-communities/socio-economic-indexes-areas-seifa-australia/2021
SEIFA_FILES = {
    "SEIFA_2021_SA2.xlsx": "https://www.abs.gov.au/statistics/people/people-and-communities/socio-economic-indexes-areas-seifa-australia/2021/Statistical%20Area%20Level%202%2C%20Indexes%2C%20SEIFA%202021.xlsx",
}

# ABS Counts of Australian Businesses, SA2 level, by industry
# source: https://www.abs.gov.au/statistics/economy/business-indicators/counts-australian-businesses-including-entries-and-exits/jul2020-jun2024
BUSINESS_FILES = {
    "business_counts_SA2_employment_size.xlsx": "https://www.abs.gov.au/statistics/economy/business-indicators/counts-australian-businesses-including-entries-and-exits/jul2020-jun2024/8165DC08.xlsx",
    "business_counts_SA2_turnover_size.xlsx": "https://www.abs.gov.au/statistics/economy/business-indicators/counts-australian-businesses-including-entries-and-exits/jul2020-jun2024/8165DC09.xlsx",
}
output_relative_dir = '../data/'

# data output directory is
abs_output_dir = output_relative_dir + 'raw_abs'
if not os.path.exists(abs_output_dir):
    os.makedirs(abs_output_dir)


def download_files(file_dict, output_dir):
    for filename, url in file_dict.items():
        output_path = f"{output_dir}/{filename}"

         # Checks if the files have already been downloaded, skip if we've already downloaded it, 
        # every time the notebook is re-run for reproducibility checks
        if os.path.exists(output_path):
            print(f"Already downloaded: {filename}")
            continue

        print(f"Downloading {filename}...")
        urlretrieve(url, output_path)
        print(f"Completed {filename}")

download_files(SEIFA_FILES, abs_output_dir)
download_files(BUSINESS_FILES, abs_output_dir)

Completed SEIFA_2021_SA2.xlsx
Completed business_counts_SA2_employment_size.xlsx
Completed business_counts_SA2_turnover_size.xlsx


In [3]:
import pandas as pd
excel_path_seifa = f"{abs_output_dir}/SEIFA_2021_SA2.xlsx"
print(pd.ExcelFile(excel_path_seifa).sheet_names)

excel_path_emp_size = f"{abs_output_dir}/business_counts_SA2_employment_size.xlsx"
print(pd.ExcelFile(excel_path_emp_size).sheet_names) 

excel_path_turn_size = f"{abs_output_dir}/business_counts_SA2_turnover_size.xlsx"
print(pd.ExcelFile(excel_path_turn_size).sheet_names)

['Contents', 'Table 1', 'Table 2', 'Table 3', 'Table 4', 'Table 5', 'Table 6', 'Explanatory Notes']
['Contents', 'Table 1', 'Table 2', 'Table 3', 'Table 4', 'Table 5', 'Table 6', 'Further information', 'Disclaimer']
['Contents', 'Table 1', 'Table 2', 'Table 3', 'Further information', 'Disclaimer']


In [4]:
pd.read_excel(excel_path_seifa, sheet_name='Contents', header=None)

,0,1,2
0,Australian Bureau of Statistics,NaN,NaN
1,"Socio-Economic Indexes for Australia (SEIFA), ...",NaN,NaN
2,Released at 11.30am (Canberra time) 27 April 2023,NaN,NaN
3,NaN,NaN,NaN
4,NaN,Contents,NaN
5,NaN,Tables,NaN
6,NaN,1,"Statistical Area Level 2 (SA2) SEIFA Summary, ..."
7,NaN,2,Statistical Area Level 2 (SA2) Index of Relati...
8,NaN,3,Statistical Area Level 2 (SA2) Index of Relati...
9,NaN,4,Statistical Area Level 2 (SA2) Index of Econom...


In [5]:
pd.read_excel(excel_path_seifa, sheet_name='Table 1', header=None, nrows=15)

,0,1,2,3,4,5,6,7,8,9,10
0,Australian Bureau of Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"Socio-Economic Indexes for Australia (SEIFA), ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Released at 11.30am (Canberra time) 27 April 2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Table 1 Statistical Area Level 2 (SA2) SEIFA S...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,Index of Relative Socio-economic Disadvantage,NaN,Index of Relative Socio-economic Advantage and...,NaN,Index of Economic Resources,NaN,Index of Education and Occupation,NaN,NaN
5,2021 Statistical Area Level 2 (SA2) 9-Digit Code,2021 Statistical Area Level 2 (SA2) Name,Score,Decile,Score,Decile,Score,Decile,Score,Decile,Usual Resident Population
6,101021007,Braidwood,1024,6,1001,6,1027,7,1008,6,4343
7,101021008,Karabar,994,5,982,5,1000,5,967,5,8517
8,101021009,Queanbeyan,1010,5,998,6,945,3,1000,6,11342
9,101021010,Queanbeyan - East,1025,6,1015,6,969,4,1025,7,5085


In [6]:
seifa_table1 = pd.read_excel(excel_path_seifa, sheet_name='Table 1', skiprows=5)

seifa_table1.columns = [
    'SA2_CODE_2021',
    'SA2_NAME_2021',
    'IRSD_score', 'IRSD_decile',
    'IRSAD_score', 'IRSAD_decile',
    'IER_score', 'IER_decile',
    'IEO_score', 'IEO_decile',
    'usual_resident_population',
]

seifa_irsad = seifa_table1[['SA2_CODE_2021', 'SA2_NAME_2021', 'IRSAD_score', 'IRSAD_decile']]

In [7]:
seifa_irsad.tail(10)
seifa_irsad = seifa_irsad.iloc[:-2] # Drop last two rows since they are not part of the data 
seifa_irsad.tail(10)

,SA2_CODE_2021,SA2_NAME_2021,IRSAD_score,IRSAD_decile
2356,801091110,Torrens,1112,10
2357,801101135,Coombs,1117,10
2358,801101136,Denman Prospect,1174,10
2359,801101139,Wright,1142,10
2360,801111140,ACT - South West,1110,9
2361,801111141,Namadgi,932,3
2362,901011001,Christmas Island,972,4
2363,901021002,Cocos (Keeling) Islands,903,2
2364,901031003,Jervis Bay,905,2
2365,901041004,Norfolk Island,958,4


In [8]:
pd.read_excel(excel_path_seifa, sheet_name='Table 2', header=None, nrows=10)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,Australian Bureau of Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"Socio-Economic Indexes for Australia (SEIFA), ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Released at 11.30am (Canberra time) 27 April 2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Table 2 Statistical Area Level 2 (SA2) Index o...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,Ranking within Australia,NaN,NaN,NaN,Ranking within State or Territory,NaN,NaN,NaN,NaN,NaN,NaN
5,2021 Statistical Area Level 2 (SA2) 9-Digit Code,2021 Statistical Area Level 2 (SA2) Name,Usual Resident Population,Score,NaN,Rank,Decile,Percentile,NaN,State,Rank,Decile,Percentile,Minimum score for SA1s in area,Maximum score for SA1s in area,% Usual Resident Population without an SA1 lev...
6,101021007,Braidwood,4343,1024.286387,NaN,1313,6,56,NaN,NSW,368,6,59,978.874363,1096.599629,0
7,101021008,Karabar,8517,994.093724,NaN,968,5,42,NaN,NSW,273,5,44,693.049228,1111.060999,0
8,101021009,Queanbeyan,11342,1009.758824,NaN,1151,5,49,NaN,NSW,328,6,53,904.784671,1085.635078,0.001587
9,101021010,Queanbeyan - East,5085,1024.576053,NaN,1315,6,56,NaN,NSW,369,6,59,918.106079,1127.955755,0.006686


In [9]:
pd.read_excel(excel_path_seifa, sheet_name='Table 3', header=None, nrows=10)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,Australian Bureau of Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"Socio-Economic Indexes for Australia (SEIFA), ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Released at 11.30am (Canberra time) 27 April 2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Table 3 Statistical Area Level 2 (SA2) Index o...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,Ranking within Australia,NaN,NaN,NaN,Ranking within State or Territory,NaN,NaN,NaN,NaN,NaN,NaN
5,2021 Statistical Area Level 2 (SA2) 9-Digit Code,2021 Statistical Area Level 2 (SA2) Name,Usual Resident Population,Score,NaN,Rank,Decile,Percentile,NaN,State,Rank,Decile,Percentile,Minimum score for SA1s in area,Maximum score for SA1s in area,% Usual Resident Population without an SA1 lev...
6,101021007,Braidwood,4343,1000.677063,NaN,1219,6,52,NaN,NSW,306,5,49,948.45379,1072.300317,0
7,101021008,Karabar,8517,982.313373,NaN,1029,5,44,NaN,NSW,274,5,44,752.918555,1115.451518,0
8,101021009,Queanbeyan,11342,998.123224,NaN,1193,6,51,NaN,NSW,301,5,49,926.97065,1080.489718,0.001587
9,101021010,Queanbeyan - East,5085,1014.994386,NaN,1357,6,58,NaN,NSW,341,6,55,927.615874,1136.308168,0.006686


In [10]:
seifa_excluded = pd.read_excel(excel_path_seifa, sheet_name='Table 6', header=None, nrows=15)
seifa_excluded

,0,1,2,3,4,5,6
0,Australian Bureau of Statistics,NaN,NaN,NaN,NaN,NaN,NaN
1,"Socio-Economic Indexes for Australia (SEIFA), ...",NaN,NaN,NaN,NaN,NaN,NaN
2,Released at 11.30am (Canberra time) 27 April 2023,NaN,NaN,NaN,NaN,NaN,NaN
3,Table 6 Statistical Area Level 2 (SA2) Exclude...,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,Area did not receive an index score,NaN,NaN,NaN
5,2021 Statistical Area Level 2 (SA2) 9-Digit Code,2021 Statistical Area Level 2 (SA2) Name,Usual Resident Population,IRSD,IRSAD,IER,IEO
6,103031075,Wollangambe - Wollemi,0,Y,Y,Y,Y
7,107011133,Port Kembla Industrial,5,Y,Y,Y,Y
8,107021135,Illawarra Catchment Reserve,9,Y,Y,Y,Y
9,111031230,Newcastle Port - Kooragang,31,Y,Y,N,N


In [11]:
seifa_irsad.head()

,SA2_CODE_2021,SA2_NAME_2021,IRSAD_score,IRSAD_decile
0,101021007,Braidwood,1001,6
1,101021008,Karabar,982,5
2,101021009,Queanbeyan,998,6
3,101021010,Queanbeyan - East,1015,6
4,101021012,Queanbeyan West - Jerrabomberra,1107,9


NameError: name 'excel_path_poa' is not defined